##### ==================ROADMAP — FRAMEWORK DE DATA QUALITY==================


Este projeto tem como objetivo desenvolver um framework **genérico e reutilizável de Data Quality**, capaz de analisar diferentes DataFrames e fontes de dados sem depender de regras específicas de um determinado domínio.

A arquitetura foi organizada em etapas independentes, permitindo evoluir gradualmente da análise estrutural dos dados até a geração de um indicador consolidado de qualidade.

> **Princípio do projeto:** o framework deve ser capaz de analisar diferentes fontes de dados sem conhecer previamente o significado das colunas ou depender de regras específicas de negócio.

###### ETAPAS DO PROJETO

***01. SCHEMA PROFILE*** ✅  
Análise estrutural do DataFrame, contemplando tipos de dados, completude, valores NULL, valores vazios, valores distintos e cardinalidade.

***02. DISTRIBUTION PROFILE***  
Análise da distribuição dos valores, frequência, concentração, diversidade e comportamento predominante das colunas.

***03. PATTERN PROFILE***  
Identificação automática de padrões nos dados, como comprimento, estrutura, composição de caracteres e padrões predominantes, sem depender do significado da coluna.

***04. DUPLICITY PROFILE*** 
Análise de duplicidade e unicidade dos registros e identificação de possíveis comportamentos de repetição nos dados.

***05. CONSISTENCY PROFILE*** 
Análise da consistência entre valores e colunas, buscando identificar comportamentos contraditórios ou relações inconsistentes dentro do próprio dataset.

***06. OUTLIER PROFILE***  
Identificação de valores ou comportamentos estatisticamente fora do padrão esperado, utilizando técnicas de análise de distribuição e detecção de anomalias.

***07. CORRELATION PROFILE***  
Análise de relações, associações e possíveis dependências entre as variáveis disponíveis no DataFrame.

***08. DATA QUALITY SCORE***  
Consolidação dos indicadores produzidos nas etapas anteriores em uma visão geral de qualidade, permitindo acompanhar o nível de qualidade do dataset de forma objetiva.

###### EVOLUÇÃO DO FRAMEWORK

O desenvolvimento seguirá uma abordagem incremental:

***ESTRUTURA → DISTRIBUIÇÃO → PADRÕES → DUPLICIDADE → CONSISTÊNCIA → ANOMALIAS → RELACIONAMENTOS → SCORE***

Cada etapa deverá gerar informações que possam ser utilizadas como entrada para as etapas seguintes, mantendo o framework ***genérico, modular, reutilizável e independente da fonte de dados***.

###### COMENTÁRIOS NO SCRIPT

O comentário presente após o título de referência da célula segue a estruturação explicativa:

- O QUE FAZ:
- COMO FAZ:
- POR QUE É IMPORTANTE:
- PERGUNTA RESPONDIDA:

###### DataFrame — Ocorrências Emergenciais da Rede de Distribuição 2026

Para este projeto, foi utilizado um conjunto de dados disponibilizado pela ***ANEEL — Agência Nacional de Energia Elétrica***, contendo informações sobre ocorrências emergenciais na rede de distribuição de energia elétrica durante o ano de 2026.

O DataFrame utilizado nesta etapa serve como ***fonte de dados para demonstração e validação do framework de Data Quality***.

###### Arquitetura Genérica

A célula de carregamento dos dados foi desenvolvida de forma independente das etapas de análise. Dessa forma, a fonte de dados pode ser substituída sem a necessidade de alterar o código das etapas seguintes.

A proposta é que, em uma versão futura, a primeira célula seja substituída por uma estrutura genérica capaz de receber diferentes fontes de dados, como:

- Arquivos CSV;
- Arquivos Parquet;
- Tabelas Delta;
- Tabelas SQL;
- Volumes do Databricks;
- Outras fontes compatíveis com Apache Spark.

Após o carregamento, o DataFrame passa a ser utilizado como entrada para as etapas de ***Schema Profile, Data Quality, análise de padrões, relevância e correlação***, mantendo o restante do notebook independente da origem dos dados.

> ***Princípio do projeto:*** a fonte de dados deve ser substituível sem alterar a lógica de análise e qualidade.

##### Bibliotecas e Dataframe

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType
from pyspark.sql import Window
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from math import ceil
from matplotlib.patches import Patch

In [0]:
# ============================================================
# 1. CARREGAMENTO DA FONTE DE DADOS
# ============================================================

# FONTE:
# Carrega o conjunto de dados de Ocorrências Emergenciais da
# Rede de Distribuição de Energia Elétrica disponibilizado pela ANEEL.

# OBJETIVO:
# Disponibilizar o DataFrame que será utilizado como entrada
# para as etapas de análise e avaliação de Data Quality.

# ARQUIVO:
# O dataset está armazenado em formato CSV no Volume do Databricks.

path = "/Volumes/setor_eletrico/distribuicao/ocorrencias_emergenciais_setor_eletrico/ocorrencias-emergenciais-rede-distribuicao-2026.csv"


# ============================================================
# 2. LEITURA DO DATASET
# ============================================================

# HEADER:
# Define a primeira linha do arquivo como nome das colunas.

# DELIMITER:
# Utiliza ponto e vírgula (;) como separador dos campos,
# conforme o formato original do arquivo.

# QUOTE:
# Define aspas duplas (") como delimitador de textos,
# permitindo interpretar corretamente valores que possam
# conter caracteres especiais ou separadores.

# INFERSCHEMA:
# Permite que o Spark identifique automaticamente os tipos
# de dados das colunas durante a leitura do arquivo.

df = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("quote", '"')
    .option("inferSchema", "true")
    .csv(path)
)


# ============================================================
# 3. VALIDAÇÃO INICIAL
# ============================================================

# EXIBIÇÃO:
# Apresenta uma amostra dos dados para validar visualmente
# se o arquivo foi carregado corretamente antes de iniciar
# as análises de qualidade.

display(
    df
)


##### 01. SCHEMA PROFILE - ANÁLISE DA DISTRIBUIÇÃO DOS DADOS

O ***Schema Profile*** é a primeira etapa do processo de análise de qualidade de dados. Seu objetivo é realizar uma avaliação estrutural do DataFrame, identificando características importantes de cada coluna antes da aplicação de regras de negócio ou análises estatísticas mais avançadas.

Nesta etapa são analisados aspectos como ***tipo de dado, completude, valores nulos, valores vazios, quantidade de valores distintos e cardinalidade***. Essas informações permitem identificar rapidamente possíveis problemas de preenchimento, colunas de baixa ou alta variabilidade e comportamentos que merecem investigação.

> ***Importante:*** o Schema Profile não determina sozinho se um dado está correto ou incorreto. Ele identifica padrões e comportamentos que servirão de base para as próximas etapas de Data Quality, como análise de padrões, validade, duplicidade, consistência, relevância e correlação.

---

###### Objetivos desta etapa

- Identificar a estrutura do DataFrame;
- Quantificar registros e colunas;
- Avaliar o preenchimento das informações;
- Identificar valores `NULL` e vazios;
- Medir a diversidade dos valores;
- Identificar colunas de baixa e alta cardinalidade;
- Detectar colunas que merecem investigação;
- Criar uma base estruturada para as próximas análises de qualidade.

###### Etapas realizadas:
- ***Schema da tabela***: mostra os tipos de dados de cada coluna, permitindo verificar se estão coerentes com o conteúdo esperado.  
- ***Resumo estatístico***: fornece métricas como mínimo, máximo, média e desvio padrão para colunas numéricas.  
- ***Contagem de nulos***: avalia a completude dos dados, identificando colunas com ausência de valores.  
- ***Cardinalidade***: mede a diversidade de valores em cada coluna, útil para detectar atributos pouco informativos ou com excesso de variação.  
- ***Distribuição de valores***: ajuda a visualizar padrões e identificar possíveis outliers ou inconsistências.  

Esse perfilamento inicial é essencial em ***engenharia de dados*** e ***qualidade da informação***, pois fornece insumos para:
- Definir regras de validação e limpeza.  
- Avaliar relevância das colunas para análises estatísticas ou modelos de machine learning.  
- Garantir integridade e consistência antes de avançar para etapas de transformação e modelagem.  


In [0]:
# ============================================================
# 1. VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame está disponível e possui colunas para que o processo de Data Quality possa ser executado.

# COMO FAZ:
# Verifica se o DataFrame foi definido e se possui pelo menos uma coluna disponível para análise.

# POR QUE É IMPORTANTE:
# Evita a execução das etapas seguintes sobre um DataFrame inexistente ou sem estrutura para análise.

# PERGUNTA RESPONDIDA:
# O DataFrame está pronto para ser analisado?

if df is None:
    raise ValueError("O DataFrame não foi definido.")

if len(df.columns) == 0:
    raise ValueError("O DataFrame não possui colunas.")

In [0]:
# ============================================================
# 2. INFORMAÇÕES GERAIS
# ============================================================

# O QUE FAZ:
# Obtém as informações básicas sobre o tamanho e a estrutura do DataFrame.

# COMO FAZ:
# Calcula a quantidade total de registros e a quantidade total de colunas existentes na tabela.

# POR QUE É IMPORTANTE:
# Define a dimensão do dataset e fornece a base necessária para os cálculos das métricas de qualidade.

# PERGUNTA RESPONDIDA:
# Qual é o tamanho e a estrutura básica do dataset?

total_registros = df.count()
total_colunas = len(df.columns)

In [0]:
# ============================================================
# 3. CONSTRUÇÃO DAS MÉTRICAS
# ============================================================

# O QUE FAZ:
# Define automaticamente as métricas que serão utilizadas para avaliar a qualidade de cada coluna.

# COMO FAZ:
# Percorre todas as colunas do DataFrame e calcula a quantidade de valores NULL, valores vazios e valores distintos.

# POR QUE É IMPORTANTE:
# Permite que o Schema Profile seja utilizado de forma genérica em diferentes tabelas sem depender de nomes de colunas previamente definidos.

# PERGUNTA RESPONDIDA:
# Quais características básicas de qualidade devem ser avaliadas em cada coluna?

metricas = []

for campo in df.schema.fields:

    nome_coluna = campo.name

    coluna = F.col(nome_coluna)
    coluna_string = coluna.cast("string")

    metricas.extend([

        # Valores NULL
        F.sum(
            F.when(coluna.isNull(), 1).otherwise(0)
        ).alias(f"{nome_coluna}__null"),

        # Valores vazios
        F.sum(
            F.when(
                coluna.isNotNull() &
                (F.trim(coluna_string) == ""),
                1
            ).otherwise(0)
        ).alias(f"{nome_coluna}__vazio"),

        # Valores distintos
        F.approx_count_distinct(
            coluna
        ).alias(f"{nome_coluna}__distintos")
    ])

In [0]:
# ============================================================
# 4. EXECUÇÃO DO PROFILE
# ============================================================

# O QUE FAZ:
# Executa as métricas definidas anteriormente sobre todos os registros do DataFrame.

# COMO FAZ:
# Realiza as agregações necessárias para obter os resultados das métricas de cada coluna.

# POR QUE É IMPORTANTE:
# Centraliza a execução das métricas e permite obter uma visão consolidada da qualidade estrutural da tabela.

# PERGUNTA RESPONDIDA:
# Quais são os resultados das métricas de qualidade calculadas para esta tabela?

resultado = df.agg(*metricas).collect()[0]

In [0]:
# ============================================================
# 5. CONSTRUÇÃO DO RESULTADO
# ============================================================

# O QUE FAZ:
# Transforma os resultados brutos das métricas em indicadores de qualidade mais fáceis de interpretar.

# COMO FAZ:
# Calcula a quantidade e o percentual de valores preenchidos, NULL, vazios, não preenchidos e distintos para cada coluna.

# POR QUE É IMPORTANTE:
# Permite comparar o comportamento das colunas utilizando indicadores absolutos e relativos, independentemente do tamanho da tabela.

# PERGUNTA RESPONDIDA:
# Qual é o nível de preenchimento e diversidade de cada coluna?

profile = []

for campo in df.schema.fields:

    nome_coluna = campo.name
    tipo_dado = campo.dataType.simpleString()

    qtd_null = resultado[f"{nome_coluna}__null"] or 0
    qtd_vazio = resultado[f"{nome_coluna}__vazio"] or 0
    qtd_distintos = resultado[f"{nome_coluna}__distintos"] or 0

    qtd_nao_preenchidos = qtd_null + qtd_vazio

    qtd_preenchidos = (
        total_registros - qtd_nao_preenchidos
    )

    pct_null = (
        qtd_null / total_registros * 100
        if total_registros > 0 else 0
    )

    pct_vazio = (
        qtd_vazio / total_registros * 100
        if total_registros > 0 else 0
    )

    pct_preenchido = (
        qtd_preenchidos / total_registros * 100
        if total_registros > 0 else 0
    )

    pct_distintos = (
        qtd_distintos / total_registros * 100
        if total_registros > 0 else 0
    )

    profile.append((
        nome_coluna,
        tipo_dado,
        total_registros,
        qtd_preenchidos,
        qtd_null,
        qtd_vazio,
        qtd_nao_preenchidos,
        qtd_distintos,
        round(pct_preenchido, 2),
        round(pct_null, 2),
        round(pct_vazio, 2),
        round(pct_distintos, 2)
    ))

In [0]:
# ============================================================
# 6. DATAFRAME DO SCHEMA PROFILE
# ============================================================

# O QUE FAZ:
# Cria um DataFrame consolidado contendo todas as métricas calculadas para cada coluna.

# COMO FAZ:
# Organiza os resultados do profile em uma estrutura tabular com uma linha para cada coluna analisada.

# POR QUE É IMPORTANTE:
# O schema_profile se torna a principal fonte para as análises, visualizações e classificações realizadas nas etapas seguintes.

# PERGUNTA RESPONDIDA:
# Como está a qualidade estrutural de cada coluna?

schema_profile = spark.createDataFrame(
    profile,
    [
        "coluna",
        "tipo_dado",
        "total_registros",
        "qtd_preenchidos",
        "qtd_null",
        "qtd_vazio",
        "qtd_nao_preenchidos",
        "qtd_distintos",
        "pct_preenchido",
        "pct_null",
        "pct_vazio",
        "pct_distintos"
    ]
)

In [0]:
# ============================================================
# 7. EXIBIÇÃO
# ============================================================

# O QUE FAZ:
# Apresenta os resultados consolidados do Schema Profile para análise exploratória.

# COMO FAZ:
# Ordena as colunas pelo percentual de preenchimento, apresentando primeiro aquelas com menor completude.

# POR QUE É IMPORTANTE:
# Facilita a identificação rápida das colunas com maior ausência de dados e dos principais pontos de atenção.

# PERGUNTA RESPONDIDA:
# Quais colunas apresentam os maiores problemas de completude?

display(
    schema_profile.orderBy(
        F.col("pct_preenchido").asc()
    )
)

In [0]:
# ============================================================
# 8. VISUALIZAÇÃO - PREENCHIMENTO X NULL
# ============================================================

# O QUE FAZ:
# Apresenta visualmente a relação entre valores preenchidos e valores NULL existentes em cada coluna.

# COMO FAZ:
# Utiliza os percentuais de preenchimento e NULL calculados pelo Schema Profile para construir a visualização.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente quais colunas apresentam maior ausência de informação.

# PERGUNTA RESPONDIDA:
# Quais colunas apresentam maior ausência de dados?

schema_profile_preenchimento = (
    schema_profile
    .select(
        "coluna",
        "pct_preenchido",
        "pct_null"
    )
    .orderBy(
        F.col("pct_preenchido").asc()
    )
)

# Transformar para formato empilhado
schema_profile_stacked = (
    schema_profile_preenchimento
    .selectExpr(
        "coluna",
        "pct_preenchido as percentual",
        "'Preenchido' as tipo"
    )
    .union(
        schema_profile_preenchimento.selectExpr(
            "coluna",
            "pct_null as percentual",
            "'Nulo' as tipo"
        )
    )
)

display(
    schema_profile_stacked
)

Databricks visualization. Run in Databricks to view.

In [0]:
# ============================================================
# 9. VISUALIZAÇÃO - CARDINALIDADE
# ============================================================

# O QUE FAZ:
# Apresenta a proporção de valores distintos existentes em relação ao total de registros de cada coluna.

# COMO FAZ:
# Utiliza o percentual de valores distintos calculado durante a construção do Schema Profile.

# POR QUE É IMPORTANTE:
# A cardinalidade ajuda a identificar o comportamento estrutural das colunas, diferenciando campos com baixa e alta variabilidade.

# PERGUNTA RESPONDIDA:
# Quanto os valores de cada coluna variam dentro da tabela?

schema_profile_cardinalidade = (
    schema_profile
    .select(
        "coluna",
        "qtd_distintos",
        "pct_distintos"
    )
    .orderBy(
        F.col("pct_distintos").desc()
    )
)

display(
    schema_profile_cardinalidade
)

In [0]:
# ============================================================
# 10. CLASSIFICAÇÃO ESTRUTURAL DAS COLUNAS
# ============================================================

# O QUE FAZ:
# Classifica as colunas de acordo com o comportamento combinado de completude e cardinalidade.

# COMPLETUDE:
# Representa o percentual de registros que possuem um valor preenchido em determinada coluna.
# Uma completude alta indica que a maior parte dos registros possui informação disponível na coluna.
# Uma completude baixa indica uma quantidade significativa de registros sem informação, o que pode representar uma ausência esperada ou um possível problema de qualidade que deve ser investigado.

# CARDINALIDADE:
# Representa a proporção de valores distintos existentes em uma coluna em relação ao total de registros.
# Uma cardinalidade baixa indica que a coluna possui poucos valores diferentes e pode representar categorias,classificações ou domínios controlados. Uma cardinalidade alta indica grande diversidade de valores e pode representar identificadores, chaves, timestamps ou campos com alta variabilidade.

# COMO FAZ:
# Aplica regras sobre os percentuais de preenchimento e valores distintos para identificar diferentes perfis estruturais.

# POR QUE É IMPORTANTE:
# Permite transformar métricas isoladas em uma primeira interpretação sobre o comportamento de cada coluna e identificar campos que merecem investigação.

# PERGUNTA RESPONDIDA:
# Que comportamento estrutural cada coluna apresenta?

schema_profile_estrutura = (
    schema_profile
    .select(
        "coluna",
        "pct_preenchido",
        "pct_distintos"
    )
    .withColumn(
        "perfil_estrutural",

        F.when(
            (F.col("pct_preenchido") >= 95) &
            (F.col("pct_distintos") >= 80),
            "Alta completude / Alta cardinalidade"
        )

        .when(
            (F.col("pct_preenchido") >= 95) &
            (F.col("pct_distintos") < 10),
            "Alta completude / Baixa cardinalidade"
        )

        .when(
            (F.col("pct_preenchido") < 50) &
            (F.col("pct_distintos") < 10),
            "Baixa completude / Baixa cardinalidade"
        )

        .otherwise(
            "Comportamento intermediário"
        )
    )
)

display(
    schema_profile_estrutura
        .orderBy(
            F.col("pct_preenchido").asc()
        )
)

##### 02. DISTRIBUTION PROFILE - ANÁLISE DE DISTRIBUIÇÃO DE VALORES, DIVERSIDADE E COMPORTAMENTO PREDOMINANTE DE COLUNAS.

O *Distribution Profile* tem como objetivo analisar como os valores estão distribuídos dentro de cada coluna do DataFrame.

Enquanto o *Schema Profile* avalia características estruturais — como completude, valores NULL, valores vazios, quantidade de distintos e cardinalidade — esta etapa busca compreender a **frequência e a concentração dos valores**.

###### O QUE É DISTRIBUIÇÃO?

Distribuição representa a forma como os valores de uma coluna estão distribuídos entre os registros da tabela.

Uma coluna pode apresentar 100% de preenchimento e possuir poucos valores responsáveis pela maior parte dos registros. Da mesma forma, outra coluna pode apresentar os mesmos 100% de preenchimento, mas possuir uma grande diversidade de valores com baixa concentração.

O *Distribution Profile* permite identificar essas diferenças de comportamento.

###### O QUE SERÁ ANALISADO?

Nesta etapa serão analisados:

- Frequência dos valores;
- Quantidade de valores distintos;
- Valores mais frequentes;
- Participação percentual dos valores mais frequentes;
- Nível de concentração dos valores;
- Diversidade dos valores (entropia);
- Identificação de valores dominantes.

###### FREQUÊNCIA DOS VALORES

A frequência representa quantas vezes determinado valor aparece dentro de uma coluna.  
Essa análise permite identificar valores predominantes e compreender quais informações representam a maior parte dos registros.

###### CONCENTRAÇÃO DOS VALORES

A concentração representa quanto dos registros está concentrado em determinados valores.  
Uma coluna pode possuir poucos valores distintos e apresentar uma distribuição equilibrada entre eles. Da mesma forma, pode possuir poucos valores distintos e ter quase todos os registros concentrados em apenas um valor.  
Essa diferença é relevante para compreender o comportamento real dos dados.

###### DIVERSIDADE DOS VALORES

A diversidade é medida pela **Entropia de Shannon**, que avalia o grau de dispersão dos registros entre diferentes valores.  
Quanto maior a entropia, maior a diversidade; quanto menor, maior a concentração em poucos valores.

###### VALORES DOMINANTES

São valores que representam uma parcela excessivamente elevada dos registros de uma coluna.  
A identificação de valores dominantes é feita com base em um limiar dinâmico, calculado a partir da distribuição dos percentuais observados.

###### POR QUE É IMPORTANTE?

A análise de distribuição complementa o *Schema Profile* porque permite identificar comportamentos que não são evidentes apenas pela quantidade de valores preenchidos ou distintos.  
Por exemplo, duas colunas podem apresentar 100% de completude, mas possuir comportamentos completamente diferentes:

***Coluna A***
- Poucos valores distintos;
- Alta concentração em um único valor.

***Coluna B***
- Muitos valores distintos;
- Baixa concentração em cada valor.

A análise de distribuição permite identificar essa diferença de comportamento.

###### ABORDAGEM GENÉRICA

Esta etapa não depende do significado das colunas e não utiliza regras específicas de negócio.  
O framework não precisa saber se uma coluna representa cliente, produto, ocorrência, cidade, código, valor ou qualquer outro atributo.  
A análise considera apenas o comportamento observado nos valores existentes no DataFrame.

###### RELAÇÃO COM O SCHEMA PROFILE

O *Schema Profile* responde principalmente:

***"Como está estruturada cada coluna?"***

O *Distribution Profile* complementa essa análise respondendo:

***"Como os valores estão distribuídos dentro de cada coluna?"***

###### PERGUNTA PRINCIPAL

***Como os valores estão distribuídos, diversificados e concentrados dentro de cada coluna?***

###### PRÓXIMA ETAPA

Os resultados desta análise serão utilizados como base para o ***03. PATTERN PROFILE***, que irá aprofundar a análise sobre os padrões estruturais presentes nos valores, mantendo o framework independente do significado das colunas.


In [0]:
# ============================================================
# 11. RESUMO DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Complementa o Schema Profile com informações relacionadas ao comportamento e à distribuição dos valores das colunas.

# COMO FAZ:
# Utiliza as informações já calculadas no Schema Profile e acrescenta métricas específicas de distribuição.

# POR QUE É IMPORTANTE:
# Permite analisar não apenas quanto uma coluna está preenchida, mas também como seus valores estão distribuídos.

# PERGUNTA RESPONDIDA:
# Como os valores estão distribuídos dentro de cada coluna?


distribution_profile = (
    schema_profile
    .select(
        "coluna",
        "tipo_dado",
        "total_registros",
        "qtd_preenchidos",
        "qtd_distintos",
        "pct_distintos"
    )
)

display(
    distribution_profile
)

In [0]:
# ============================================================
# 12. VALORES MAIS FREQUENTES
# ============================================================

# O QUE FAZ:
# Calcula a quantidade de ocorrências de cada valor existente nas colunas do DataFrame.

# COMO FAZ:
# Agrupa os valores individualmente por coluna e contabiliza suas respectivas ocorrências.

# POR QUE É IMPORTANTE:
# Permite identificar quais valores aparecem com maior frequência e compreender o comportamento da distribuição de cada coluna.

# PERGUNTA RESPONDIDA:
# Quantas vezes cada valor aparece em cada coluna?

frequencias = []

for campo in df.schema.fields:
    nome_coluna = campo.name

    # Correção de valores ",00" para "0,00"
    coluna_corrigida = F.when(
        F.col(nome_coluna).cast("string") == ",00",
        "0,00"
    ).otherwise(F.col(nome_coluna).cast("string"))

    frequencia_coluna = (
        df
        .select(
            F.lit(nome_coluna).alias("coluna"),
            coluna_corrigida.alias("valor")
        )
        .where(F.col(nome_coluna).isNotNull())
        .groupBy("coluna", "valor")
        .count()
    )

    frequencias.append(frequencia_coluna)

#Unir todos os resultados em um único DataFrame
frequencia_profile = frequencias[0]
for frequencia_coluna in frequencias[1:]:
    frequencia_profile = frequencia_profile.unionByName(frequencia_coluna)

#Opcional: ordenar para facilitar leitura
frequencia_profile = frequencia_profile.orderBy(F.desc("count"))

display(frequencia_profile)

In [0]:
# ============================================================
# 13. TOP VALORES MAIS FREQUENTES
# ============================================================

# O QUE FAZ:
# Identifica os valores mais recorrentes em cada coluna, ordenando pela frequência. Ordena o resultado final pela contagem (maior para menor).

# COMO FAZ:
# Para cada coluna, ordena os valores pela contagem e seleciona os Top N. Depois une todos os resultados e aplica ordenação global.

# POR QUE É IMPORTANTE:
# Permite identificar valores dominantes e compreender se a distribuição é concentrada em poucos valores.

# PERGUNTA RESPONDIDA:
# Quais são os valores mais frequentes em cada coluna?

TOP_N = 1

top_valores = (
    frequencia_profile
    .withColumn(
        "rank",
        F.row_number().over(Window.partitionBy("coluna").orderBy(F.desc("count")))
    )
    .where(F.col("rank") <= TOP_N)
    .orderBy(F.desc("count"))
    .select("coluna", "valor", "count")
)

display(top_valores)

In [0]:
# ============================================================
# 14. PARTICIPAÇÃO PERCENTUAL DOS VALORES
# ============================================================

# O QUE FAZ:
# Calcula o percentual de participação de cada valor dentro da coluna.

# COMO FAZ:
# Soma o total de registros por coluna e divide a contagem de cada valor por esse total. Ordena os resultados para facilitar a leitura.

# POR QUE É IMPORTANTE:
# Permite comparar colunas de diferentes tamanhos de forma estatística, destacando a concentração relativa dos valores.

# PERGUNTA RESPONDIDA:
# Qual a participação relativa dos valores mais frequentes em cada coluna?


#Totais por coluna
totais_coluna = (
    frequencia_profile
    .groupBy("coluna")
    .agg(F.sum("count").alias("total_coluna"))
)

#Junta os totais e calcula percentual
frequencia_percentual = (
    frequencia_profile
    .join(totais_coluna, on="coluna", how="left")
    .withColumn("pct_valor", F.round((F.col("count") / F.col("total_coluna")) * 100, 4))
    .orderBy(F.desc("pct_valor"))
    .select("coluna", "valor", "count", "pct_valor")
)

display(frequencia_percentual)


In [0]:
# ============================================================
# 15. TOP 5 VALORES POR COLUNA (percentual proporcional)
# ============================================================

# O QUE FAZ:
# Seleciona os 5 valores mais representativos de cada coluna e recalcula o percentual de participação proporcional a 100% dentro de cada coluna.

# COMO FAZ:
# Usa Window para ranquear os valores por percentual dentro de cada coluna. Filtra os Top 5 e recalcula o percentual relativo considerando apenas os valores daquela coluna.

# POR QUE É IMPORTANTE:
# Permite visualizar os principais valores de cada coluna de forma comparável, garantindo que os percentuais somem 100% dentro de cada atributo.

# PERGUNTA RESPONDIDA:
# Quais são os 5 valores mais representativos em cada coluna e qual sua participação proporcional dentro dela?

# Janela para ranquear os valores por coluna
window_top = (
    Window
    .partitionBy("coluna")
    .orderBy(F.desc("pct_valor"))
)

# Seleção dos Top 5 valores por coluna
frequencia_top5 = (
    frequencia_percentual
    .withColumn("ranking", F.row_number().over(window_top))
    .filter(F.col("ranking") <= 5)
)

# Recalcula percentuais proporcionais a 100% dentro de cada coluna
totais_top5 = (
    frequencia_top5
    .groupBy("coluna")
    .agg(F.sum("pct_valor").alias("total_pct_coluna"))
)

frequencia_top5 = (
    frequencia_top5
    .join(totais_top5, on="coluna", how="left")
    .withColumn(
        "pct_proporcional",
        F.round((F.col("pct_valor") / F.col("total_pct_coluna")) * 100, 2)
    )
    .orderBy("coluna", "ranking")
    .select("coluna", "valor", "count", "pct_valor", "pct_proporcional", "ranking")
)

display(frequencia_top5)

In [0]:
# ============================================================
# 16. NÍVEL DE CONCENTRAÇÃO DOS VALORES MAIS FREQUENTES POR COLUNA
# ============================================================

# O QUE FAZ:
# Mede quanto dos registros de cada coluna está concentrado nos valores mais frequentes, utilizando os indicadores Top 1, Top 3 e Top 5.

# COMO FAZ:
# Primeiro, os valores de cada coluna são ordenados de acordo com seu percentual de participação.
#
# Top 1: representa a participação do valor mais frequente da coluna.
# Top 3: representa a soma da participação dos três valores mais frequentes.
# Top 5: representa a soma da participação dos cinco valores mais frequentes.
#
# Dessa forma, quanto maior o percentual acumulado no Top 1, Top 3 ou Top 5, maior é a concentração dos registros em poucos valores.

# POR QUE É IMPORTANTE:
# Permite identificar colunas dominadas por poucos valores e avaliar o nível de concentração da distribuição.
# Uma concentração elevada não representa necessariamente um problema de qualidade, pois pode ser característica natural da variável. Entretanto, valores excessivamente concentrados podem indicar baixa diversidade, categorias dominantes ou possíveis padrões que precisam ser investigados em análises posteriores.

# PERGUNTA RESPONDIDA:
# Quanto dos registros está concentrado nos valores mais frequentes
# de cada coluna?


concentracao_profile = (
    frequencia_top5
    .groupBy("coluna")
    .agg(
        F.round(
            F.sum(F.when(F.col("ranking") <= 1, F.col("pct_valor")).otherwise(0)),
            4
        ).alias("concentracao_top1"),

        F.round(
            F.sum(F.when(F.col("ranking") <= 3, F.col("pct_valor")).otherwise(0)),
            4
        ).alias("concentracao_top3"),

        F.round(
            F.sum(F.when(F.col("ranking") <= 5, F.col("pct_valor")).otherwise(0)),
            4
        ).alias("concentracao_top5")
    )
    .orderBy(F.desc("concentracao_top1"))
)

display(concentracao_profile)

In [0]:
# ============================================================
# 17. ÍNDICE DE DIVERSIDADE DOS VALORES POR COLUNA
# ============================================================

# O QUE FAZ:
# Calcula o índice de diversidade de cada coluna utilizando a Entropia de Shannon, considerando a distribuição dos valores presentes na coluna.

# COMO FAZ:
# Utiliza o percentual de participação de cada valor, calculado anteriormente no DataFrame frequencia_percentual, para obter a probabilidade de ocorrência de cada valor.
# A Entropia de Shannon é calculada pela fórmula:
#
# H = -Σ(p × ln(p))
#
# p: representa a proporção de ocorrência de cada valor na coluna.
#
# Quanto maior a entropia, maior tende a ser a diversidade e a distribuição dos registros entre diferentes valores.
# Quanto menor a entropia, maior tende a ser a concentração dos registros em poucos valores.
# Como a Entropia de Shannon é influenciada pela quantidade de valores existentes na coluna, também é calculada a Entropia Normalizada:
#
# Entropia Normalizada = Entropia de Shannon / ln(cardinalidade)
#
# A Entropia Normalizada permite uma comparação mais adequada entre colunas com diferentes níveis de cardinalidade, variando aproximadamente entre 0 e 1.
#
# Para facilitar a interpretação, é criada uma leitura da diversidade:
#
# 0,00 a 0,20 - Muito baixa diversidade
# 0,20 a 0,40 - Baixa diversidade
# 0,40 a 0,60 - Diversidade moderada
# 0,60 a 0,80 - Alta diversidade
# 0,80 a 1,00 - Muito alta diversidade
#
# Esses limites são utilizados como referência analítica para o notebook e não representam limites estatísticos universais.

# POR QUE É IMPORTANTE:
# Permite avaliar a diversidade da distribuição dos valores de cada coluna por meio de uma métrica estatística e facilita a comparação entre colunas com diferentes cardinalidades.
# Uma diversidade muito baixa pode indicar forte concentração dos registros em poucos valores, enquanto uma diversidade elevada indica uma distribuição mais dispersa entre os valores existentes.
# A leitura deve ser analisada em conjunto com a cardinalidade e o nível de concentração, pois uma baixa diversidade não representa necessariamente um problema de qualidade. Algumas colunas possuem naturalmente poucos valores ou valores dominantes.

# PERGUNTA RESPONDIDA:
# Quão diversificados estão os valores presentes em cada coluna?

diversidade_profile = (
    frequencia_percentual
    # Converte o percentual em probabilidade (p = pct_valor / 100)
    .withColumn(
        "probabilidade",
        F.col("pct_valor") / 100
    )
    # Calcula a entropia parcial de cada valor: -(p * ln(p))
    .withColumn(
        "entropia_parcial",
        -(F.col("probabilidade") * F.log(F.col("probabilidade")))
    )
    # Agrupa por coluna para calcular métricas globais
    .groupBy("coluna")
    .agg(
        # Cardinalidade: número de valores distintos na coluna
        F.countDistinct("valor").alias("cardinalidade"),
        # Entropia de Shannon: soma das entropias parciais
        F.round(F.sum("entropia_parcial"), 4).alias("entropia_shannon")
    )
    # Entropia máxima: ln(cardinalidade), usada para normalização
    .withColumn(
        "entropia_maxima",
        F.when(F.col("cardinalidade") > 1, F.log(F.col("cardinalidade"))).otherwise(0)
    )
    # Entropia normalizada: Shannon / máxima (varia entre 0 e 1)
    .withColumn(
        "entropia_normalizada",
        F.when(
            F.col("entropia_maxima") > 0,
            F.round(F.col("entropia_shannon") / F.col("entropia_maxima"), 4)
        ).otherwise(0)
    )
    # Classificação qualitativa da diversidade
    .withColumn(
        "leitura_diversidade",
        F.when(F.col("entropia_normalizada") <= 0.20, "Muito baixa diversidade")
        .when(F.col("entropia_normalizada") <= 0.40, "Baixa diversidade")
        .when(F.col("entropia_normalizada") <= 0.60, "Diversidade moderada")
        .when(F.col("entropia_normalizada") <= 0.80, "Alta diversidade")
        .otherwise("Muito alta diversidade")
    )
    # Seleciona apenas as colunas finais relevantes
    .select(
        "coluna",
        "cardinalidade",
        "entropia_shannon",
        "entropia_maxima",
        "entropia_normalizada",
        "leitura_diversidade"
    )
    # Ordena pela entropia normalizada (colunas mais diversas primeiro)
    .orderBy(F.desc("entropia_normalizada"))
)

# Exibe o resultado
display(diversidade_profile)

In [0]:
# ============================================================
# 18. IDENTIFICAÇÃO DE VALORES DOMINANTES (limiar dinâmico)
# ============================================================

# O QUE FAZ:
# Analisa a distribuição dos percentuais de participação dos valores em todas as colunas e define automaticamente um limiar de dominância (ex.: percentil 95).
# Valores acima desse limiar são considerados dominantes.

# COMO FAZ:
# - Calcula o percentil 95 dos percentuais (pct_valor) em todo o dataset.
# - Usa esse valor como limiar_dominancia.
# - Filtra os valores que ultrapassam esse limiar e marca como "dominante".

# POR QUE É IMPORTANTE:
# Evita definir um limite fixo e permite que o critério de dominância seja adaptado ao comportamento real dos dados.

# PERGUNTA RESPONDIDA:
# Quais valores ultrapassam o limiar estatístico de dominância em cada coluna?

# Calcula concentração Top 1 por coluna
concentracao_top1 = (
    frequencia_percentual
    .withColumn(
        "ranking",
        F.row_number().over(Window.partitionBy("coluna").orderBy(F.desc("pct_valor")))
    )
    .filter(F.col("ranking") == 1)
    .select("coluna", "valor", "pct_valor")
)

# Calcula estatísticas globais
stats = concentracao_top1.agg(
    F.avg("pct_valor").alias("media"),
    F.stddev("pct_valor").alias("desvio")
).collect()[0]

limiar_dominancia = stats["media"] + stats["desvio"]

# Filtra colunas dominadas
valores_dominantes = (
    concentracao_top1
    .filter(F.col("pct_valor") >= limiar_dominancia)
    .withColumn("status", F.lit("Valor dominante"))
    .orderBy(F.desc("pct_valor"))
)

display(valores_dominantes)

print(f"Limiar de dominância sugerido automaticamente: {round(limiar_dominancia,2)}%")

###### VISUALIZAÇÕES GRÁFICAS

In [0]:
# ============================================================
# 19. VISUALIZAÇÃO: TOP 5 VALORES MAIS FREQUENTES POR COLUNA
# ============================================================

# Converter para pandas
df_top5_pd = frequencia_top5.toPandas()

# Lista de colunas únicas
colunas_unicas = df_top5_pd['coluna'].unique()

# Configuração de layout: 3 gráficos por linha
n_colunas = len(colunas_unicas)
n_cols = 3
n_rows = ceil(n_colunas / n_cols)

# Paleta e estilo visual (igual à célula 30)
sns.set_style("white")
custom_palette = sns.color_palette(["#08519C", "#6BAED6", "#9ECAE1"])
sns.set_palette(custom_palette)

TITLE_SIZE = 13   # título sutil
LABEL_SIZE = 10   # rótulos discretos
TICK_SIZE = 9     # fonte menor nos ticks

# Criar figura (ajustada para ocupar todo o espaço disponível)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
fig.suptitle('Top 5 Valores Mais Frequentes por Coluna', fontsize=TITLE_SIZE, fontweight='semibold', y=0.995, color="#2C3E50")

axes = axes.flatten() if n_rows > 1 else [axes]

# Criar gráficos
for idx, coluna in enumerate(colunas_unicas):
    ax = axes[idx]
    dados_coluna = (
        df_top5_pd[df_top5_pd['coluna'] == coluna]
        .sort_values('pct_proporcional', ascending=True)
    )

    # Usar matplotlib com a mesma paleta da célula 30
    ax.barh(
        dados_coluna['valor'],
        dados_coluna['pct_proporcional'],
        color=custom_palette[0],
        alpha=0.9
    )

    # Labels sutis de percentual
    for i, (valor, pct) in enumerate(zip(dados_coluna['valor'], dados_coluna['pct_proporcional'])):
        ax.text(pct + 1, i, f'{pct:.1f}%', va='center', fontsize=TICK_SIZE, color="#34495E")

    # Configurações visuais
    ax.set_title(coluna, fontsize=LABEL_SIZE, fontweight='semibold', color="#2C3E50", pad=6)
    ax.set_xlabel('Percentual (%)', fontsize=LABEL_SIZE, color="#2C3E50")
    ax.set_ylabel('')
    ax.set_xlim(0, 105)
    ax.tick_params(axis='both', labelsize=TICK_SIZE, colors="#2C3E50")
    ax.grid(axis='x', alpha=0.15, color="#BDC3C7")

# Remover subplots vazios
for idx in range(len(colunas_unicas), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.subplots_adjust(left=0.05, right=0.98, top=0.985, bottom=0.03, hspace=0.4, wspace=0.3)
plt.show()

In [0]:
# ============================================================
# 21. VISUALIZAÇÃO: NÍVEL DE CONCENTRAÇÃO DOS VALORES POR COLUNA
# ============================================================

# Converter para pandas
df_concentracao_pd = (
    concentracao_profile
    .toPandas()
    .sort_values('concentracao_top1', ascending=False)
)

# Preparar dados
colunas = df_concentracao_pd['coluna']
top1 = df_concentracao_pd['concentracao_top1']
top3 = df_concentracao_pd['concentracao_top3']
top5 = df_concentracao_pd['concentracao_top5']

# Configurar posições das barras
x = np.arange(len(colunas))
width = 0.25

# Paleta e estilo visual
sns.set_style("white")
custom_palette = sns.color_palette(["#08519C", "#6BAED6", "#9ECAE1"])  # azul fosco suave
sns.set_palette(custom_palette)

TITLE_SIZE = 13   # título sutil
LABEL_SIZE = 10   # rótulos discretos
TICK_SIZE = 9     # fonte menor nos ticks

# Criar figura (ajustada para ocupar todo o espaço disponível)
fig, ax = plt.subplots(figsize=(16, 11))
fig.suptitle(
    'Nível de Concentração dos Valores por Coluna',
    fontsize=TITLE_SIZE, fontweight='semibold', y=0.995, color="#2C3E50"
)

# Criar barras agrupadas
bars1 = ax.bar(x - width, top1, width, label='Top 1', color=custom_palette[0], alpha=0.9)
bars2 = ax.bar(x, top3, width, label='Top 3', color=custom_palette[1], alpha=0.9)
bars3 = ax.bar(x + width, top5, width, label='Top 5', color=custom_palette[2], alpha=0.9)

# Adicionar labels sutis acima das barras
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2,
            height + 1.5,
            f'{height:.1f}%',
            ha='center', va='bottom',
            fontsize=TICK_SIZE, color="#34495E", rotation=90
        )

# Configurações visuais
ax.set_xlabel('Coluna', fontsize=LABEL_SIZE, color="#2C3E50")
ax.set_ylabel('Concentração (%)', fontsize=LABEL_SIZE, color="#2C3E50")
ax.set_xticks(x)
ax.set_xticklabels(colunas, rotation=45, ha='right', fontsize=TICK_SIZE, color="#2C3E50")
ax.legend(fontsize=LABEL_SIZE, loc='upper right', frameon=False)
ax.tick_params(axis='both', labelsize=TICK_SIZE, colors="#2C3E50")
ax.grid(axis='y', alpha=0.15, color="#BDC3C7")
ax.set_ylim(0, 110)

# Remover bordas do gráfico
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)

# Linha de referência em 50%
ax.axhline(y=50, color="#95A5A6", linestyle='--', linewidth=1, alpha=0.5)
ax.text(len(colunas)-0.5, 52, '50% referência', fontsize=TICK_SIZE, color="#7F8C8D")

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.subplots_adjust(left=0.05, right=0.98, top=0.985, bottom=0.08)
plt.show()

In [0]:
# ============================================================
# 21. VISUALIZAÇÃO: ÍNDICE DE DIVERSIDADE POR COLUNA
# ============================================================

# Converter para pandas e ordenar
df_diversidade_pd = (
    diversidade_profile
    .toPandas()
    .sort_values('entropia_normalizada', ascending=True)
)

# Usar o mesmo azul da célula 29 com transparências diferentes
sns.set_style("white")
BASE_COLOR = "#08519C"  # mesmo azul da célula 29

TITLE_SIZE = 13   # título sutil
LABEL_SIZE = 10   # rótulos discretos
TICK_SIZE = 9     # fonte menor nos ticks

# Definir transparências por faixa de diversidade (maior entropia = mais opaco)
def get_alpha(entropia):
    if entropia <= 0.20:
        return 0.2  # muito transparente (baixa diversidade)
    elif entropia <= 0.40:
        return 0.35  # transparente
    elif entropia <= 0.60:
        return 0.5  # semi-transparente
    elif entropia <= 0.80:
        return 0.7  # pouco transparente
    else:
        return 0.9  # quase opaco (alta diversidade)

alphas = df_diversidade_pd['entropia_normalizada'].apply(get_alpha)

# Criar figura (ajustada para ocupar todo o espaço disponível)
fig, ax = plt.subplots(figsize=(16, max(8, len(df_diversidade_pd) * 0.4)))
fig.suptitle(
    'Índice de Diversidade por Coluna',
    fontsize=TITLE_SIZE, fontweight='semibold', y=0.995, color="#2C3E50"
)

# Gráfico de barras horizontais com mesma cor mas diferentes transparências
for idx, (coluna, valor, alpha) in enumerate(zip(df_diversidade_pd['coluna'], df_diversidade_pd['entropia_normalizada'], alphas)):
    ax.barh(coluna, valor, color=BASE_COLOR, alpha=alpha)

# Labels sutis com valores
for i, (coluna, valor) in enumerate(zip(df_diversidade_pd['coluna'], df_diversidade_pd['entropia_normalizada'])):
    ax.text(
        valor + 0.02, i,
        f'{valor:.3f}',
        va='center', fontsize=TICK_SIZE, color="#34495E"
    )

# Configurações visuais
ax.set_xlabel('Entropia Normalizada (0 = baixa diversidade, 1 = alta diversidade)', fontsize=LABEL_SIZE, color="#2C3E50")
ax.set_ylabel('Coluna', fontsize=LABEL_SIZE, color="#2C3E50")
ax.set_xlim(0, 1.05)
ax.tick_params(axis='both', labelsize=TICK_SIZE, colors="#2C3E50")
ax.grid(axis='x', alpha=0.15, color="#BDC3C7")

# Linhas de referência discretas
for threshold in [0.20, 0.40, 0.60, 0.80]:
    ax.axvline(x=threshold, color="#95A5A6", linestyle='--', linewidth=0.8, alpha=0.5)
    ax.text(threshold, len(df_diversidade_pd) - 0.5, f'{threshold:.2f}', ha='center', fontsize=8, color="#7F8C8D")

# Legenda minimalista (mesmo azul com diferentes transparências)
import matplotlib.colors as mcolors
legend_elements = [
    Patch(facecolor=BASE_COLOR, alpha=0.2, label='Muito baixa (0.00-0.20)'),
    Patch(facecolor=BASE_COLOR, alpha=0.35, label='Baixa (0.20-0.40)'),
    Patch(facecolor=BASE_COLOR, alpha=0.5, label='Moderada (0.40-0.60)'),
    Patch(facecolor=BASE_COLOR, alpha=0.7, label='Alta (0.60-0.80)'),
    Patch(facecolor=BASE_COLOR, alpha=0.9, label='Muito alta (0.80-1.00)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=LABEL_SIZE, title='Faixas de Diversidade', frameon=False)

# Remover bordas superiores e direitas
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.subplots_adjust(left=0.05, right=0.98, top=0.985, bottom=0.08)
plt.show()


In [0]:
# ============================================================
# 23. VISUALIZAÇÃO: HEATMAP DE CONCENTRAÇÃO
# ============================================================

# O QUE FAZ:
# Cria um heatmap mostrando a concentração (Top 1, Top 3, Top 5) para cada coluna.

# Converter para pandas e ordenar
df_concentracao_pd = (
    concentracao_profile
    .toPandas()
    .sort_values('concentracao_top1', ascending=False)
)

# Preparar dados para o heatmap
data_heatmap = df_concentracao_pd[['concentracao_top1', 'concentracao_top3', 'concentracao_top5']]
colunas = df_concentracao_pd['coluna'].tolist()
metricas = ['Top 1', 'Top 3', 'Top 5']

# Estilo visual (consistente com células anteriores)
sns.set_style("white")
BASE_COLOR = "#08519C"  # azul fosco consistente

TITLE_SIZE = 13   # título sutil
LABEL_SIZE = 10   # rótulos discretos
TICK_SIZE = 9     # fonte menor nos ticks

# Criar figura (ajustada para ocupar todo o espaço disponível)
fig, ax = plt.subplots(figsize=(16, max(8, len(colunas) * 0.4)))
fig.suptitle(
    'Heatmap de Concentração dos Valores',
    fontsize=TITLE_SIZE, fontweight='semibold', y=0.995, color="#2C3E50"
)

# Criar heatmap com colormap monocromático em azul
cmap = sns.light_palette(BASE_COLOR, as_cmap=True)

# Primeiro plotar o heatmap sem anotações
sns.heatmap(
    data_heatmap,
    cmap=cmap,
    annot=False,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Percentual de Concentração (%)'},
    xticklabels=metricas,
    yticklabels=colunas,
    ax=ax,
    vmin=0, vmax=100
)

# Adicionar anotações com cores dinâmicas baseadas no valor
for i in range(len(colunas)):
    for j in range(len(metricas)):
        valor = data_heatmap.iloc[i, j]
        # Se o valor for alto (> 50%), usar branco; caso contrário, usar azul escuro
        cor_texto = 'white' if valor > 50 else BASE_COLOR
        ax.text(
            j + 0.5, i + 0.5, f'{valor:.1f}',
            ha='center', va='center',
            fontsize=TICK_SIZE, color=cor_texto, weight='semibold'
        )

# Configurações visuais
ax.set_xlabel('Métrica de Concentração', fontsize=LABEL_SIZE, color="#2C3E50")
ax.set_ylabel('Coluna', fontsize=LABEL_SIZE, color="#2C3E50")
ax.tick_params(axis='both', labelsize=TICK_SIZE, colors="#2C3E50")

# Ajustar cor da barra de cor
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=TICK_SIZE, colors="#2C3E50")
cbar.set_label('Percentual de Concentração (%)', fontsize=LABEL_SIZE, color="#2C3E50")

# Remover bordas superiores e direitas
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.subplots_adjust(left=0.08, right=0.98, top=0.96, bottom=0.05)
plt.show()


In [0]:
# ============================================================
# 24. VISUALIZAÇÃO: DIVERSIDADE VS CONCENTRAÇÃO
# ============================================================

# O QUE FAZ:
# Cria um gráfico de dispersão comparando a entropia normalizada (diversidade),

# Converter para pandas
df_diversidade_pd = diversidade_profile.toPandas()
df_concentracao_pd = concentracao_profile.toPandas()

# Combinar os dois DataFrames
df_combined = df_diversidade_pd.merge(df_concentracao_pd, on='coluna')

# Estilo visual consistente
sns.set_style("white")
BASE_COLOR = "#08519C"  # azul fosco consistente

TITLE_SIZE = 13   # título sutil
LABEL_SIZE = 10   # rótulos discretos
TICK_SIZE = 9     # fonte menor nos ticks

# Criar figura (ajustada para ocupar todo o espaço disponível)
fig, ax = plt.subplots(figsize=(16, 10))
fig.suptitle(
    'Diversidade vs Concentração por Coluna',
    fontsize=TITLE_SIZE, fontweight='semibold', y=0.995, color="#2C3E50"
)

# Scatter plot com cor única e transparência
scatter = ax.scatter(
    df_combined['entropia_normalizada'],
    df_combined['concentracao_top1'],
    s=df_combined['cardinalidade'] / df_combined['cardinalidade'].max() * 350 + 50,
    color=BASE_COLOR,
    alpha=0.6,
    edgecolors="#2C3E50",
    linewidth=0.8
)

# Linhas de referência
ax.axhline(y=50, color="#95A5A6", linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(x=0.5, color="#95A5A6", linestyle='--', linewidth=1, alpha=0.5)

# Quadrantes explicativos discretos
ax.text(0.05, 90, 'Baixa diversidade\nAlta concentração',
        fontsize=TICK_SIZE, alpha=0.7, color="#2C3E50")
ax.text(0.75, 90, 'Alta diversidade\nAlta concentração',
        fontsize=TICK_SIZE, alpha=0.7, color="#2C3E50")
ax.text(0.05, 10, 'Baixa diversidade\nBaixa concentração',
        fontsize=TICK_SIZE, alpha=0.7, color="#2C3E50")
ax.text(0.75, 10, 'Alta diversidade\nBaixa concentração',
        fontsize=TICK_SIZE, alpha=0.7, color="#2C3E50")

# Configurações visuais
ax.set_xlabel('Entropia Normalizada (Diversidade)', fontsize=LABEL_SIZE, color="#2C3E50")
ax.set_ylabel('Concentração Top 1 (%)', fontsize=LABEL_SIZE, color="#2C3E50")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-5, 105)
ax.tick_params(axis='both', labelsize=TICK_SIZE, colors="#2C3E50")
ax.grid(alpha=0.15, color="#BDC3C7")

# Remover bordas superiores e direitas
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.subplots_adjust(left=0.06, right=0.98, top=0.985, bottom=0.06)
plt.show()


##### 03. PATTERN PROFILE — IDENTIFICAÇÃO AUTOMÁTICA DE PADRÕES NOS DADOS